
# 🔗 Stage 3: Dataset Integration

### 🎯 Objective
Load the cleaned datasets into this notebook so we can integrate them safely.

We will preserve the original cleaned datasets and use separate DataFrames for integration. No records will be removed or merged in this step.


In [12]:

import os
import pandas as pd
import geopandas as gpd
import numpy as np

# Cleaned data directory
CLEANED_DIR = "../Data/processed_data/cleaned"

# CREAMS
creams_daily = pd.read_csv(
    os.path.join(CLEANED_DIR, "creams","creams_daily_clean.csv"),
    parse_dates=["date"]
)

creams_state_daily = pd.read_csv(
    os.path.join(CLEANED_DIR,"creams", "creams_state_daily_clean.csv"),
    parse_dates=["date"]
)

creams_year_compare = pd.read_csv(
    os.path.join(CLEANED_DIR, "creams","creams_year_compare_clean.csv")
)

# FIRMS
fires_daily = pd.read_csv(
    os.path.join(CLEANED_DIR, "fires", "fires_district_daily_clean.csv.gz"),
    parse_dates=["date"]
)

fires_season = pd.read_csv(
    os.path.join(CLEANED_DIR, "fires", "fires_district_season_clean.csv")
)

# Paddy
paddy_district = pd.read_csv(
    os.path.join(CLEANED_DIR, "paddy", "paddy_rice_area_district_clean.csv")
)

# Burned Area
burned_district = pd.read_csv(
    os.path.join(CLEANED_DIR, "burned_area", "burned_district_season_clean.csv")
)

burned_daily = pd.read_csv(
    os.path.join(CLEANED_DIR, "burned_area", "burned_daily_by_state_clean.csv"),
    parse_dates=["date"]
)

# Boundaries
district_boundaries = gpd.read_file(
    os.path.join(CLEANED_DIR, "boundaries", "districts_clean.geojson")
)

# Verify loading
datasets = {
    "CREAMS Daily": creams_daily,
    "CREAMS State Daily": creams_state_daily,
    "CREAMS Year Compare": creams_year_compare,
    "FIRMS Daily": fires_daily,
    "FIRMS Seasonal": fires_season,
    "Paddy District": paddy_district,
    "Burned District": burned_district,
    "Burned Daily": burned_daily,
    "District Boundaries": district_boundaries
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")


CREAMS Daily: (198423, 13)
CREAMS State Daily: (5588, 12)
CREAMS Year Compare: (5577, 10)
FIRMS Daily: (427728, 8)
FIRMS Seasonal: (84875, 8)
Paddy District: (6674, 12)
Burned District: (1265, 6)
Burned Daily: (7544, 6)
District Boundaries: (857, 6)



## 🗺️ Step 2: Inspect Geographic Identifiers

### Objective
Understand how each dataset represents districts in Punjab and Haryana.

We will inspect state labels, district names and available identifiers before deciding which columns can safely be used for integration.

No merging or name standardization will be performed yet.


In [13]:

# 1. Define a helper to inspect geographic coverage

def inspect_geography(dataset_name, df, state_col, district_col, id_columns=None):

    print("\n" + "=" * 65)
    print(dataset_name.upper())
    print("=" * 65)

    print("\nState labels:")
    print(sorted(df[state_col].dropna().astype(str).unique()))

    # Filter Punjab and Haryana
    ph_data = df[
        df[state_col].astype("string").str.strip().isin(
            ["Punjab", "Haryana"]
        )
    ].copy()

    print("\nPunjab-Haryana records:", len(ph_data))

    print("\nDistrict count by state:")
    print(ph_data.groupby(state_col)[district_col].nunique())

    print("\nDistrict names:")
    for state in ["Punjab", "Haryana"]:
        districts = sorted(
            ph_data.loc[
                ph_data[state_col] == state, district_col
            ].dropna().astype(str).unique()
        )

        print(f"\n{state} ({len(districts)} districts):")
        print(districts)

    if id_columns:
        print("\nAvailable geographic identifiers:")
        display(ph_data[id_columns].drop_duplicates().head(15))


# 2. Inspect CREAMS

inspect_geography(
    "CREAMS Daily",
    creams_daily,
    "state",
    "district"
)

# 3. Inspect FIRMS daily

inspect_geography(
    "FIRMS Daily",
    fires_daily,
    "state",
    "district",
    ["state", "district", "district_id"]
)

# 4. Inspect FIRMS seasonal

inspect_geography(
    "FIRMS Seasonal",
    fires_season,
    "state",
    "district",
    ["state", "district", "district_id"]
)

# 5. Inspect Paddy district

inspect_geography(
    "Paddy District",
    paddy_district,
    "state",
    "district",
    ["state", "district", "district_id", "lgd_code"]
)

# 6. Inspect Burned Area district

inspect_geography(
    "Burned Area District",
    burned_district,
    "state",
    "district",
    ["state", "district", "district_id"]
)

# 7. Inspect district boundaries

print("\n" + "=" * 65)
print("DISTRICT BOUNDARIES")
print("=" * 65)

ph_boundaries = district_boundaries[
    district_boundaries["state"].astype("string").str.strip().isin(
        ["Punjab", "Haryana"]
    )
]

print("Punjab-Haryana boundary records:", len(ph_boundaries))

print("\nDistrict count by state:")
print(ph_boundaries.groupby("state")["name"].nunique())

display(
    ph_boundaries[["id", "name", "state"]]
    .sort_values(["state", "name"])
    .reset_index(drop=True)
)



CREAMS DAILY

State labels:
['Delhi', 'Haryana', 'Madhya Pradesh', 'Punjab', 'Rajasthan', 'Uttar Pradesh', 'Uttarakhand']

Punjab-Haryana records: 47667

District count by state:
state
Haryana    26
Punjab     25
Name: district, dtype: int64

District names:

Punjab (25 districts):
['Amritsar', 'Barnala', 'Bathinda', 'Faridkot', 'Fatehgarh Sahib', 'Fazilka', 'Firozpur', 'Gurdaspur', 'Hoshiarpur', 'Jalandhar', 'Kapurthala', 'Kharar', 'Ludhiana', 'Malerkotla', 'Mansa', 'Moga', 'Pathankot', 'Patiala', 'Punjab', 'Rupnagar', 'SAS Nagar', 'SBS Nagar', 'Sangrur', 'Sri Muktsar Sahib', 'Tarn Taran']

Haryana (26 districts):
['Ambala', 'Bhiwani', 'Charkhi Dadri', 'Faridabad', 'Faridkot', 'Faridkot Faridabad', 'Fatehabad', 'Gurugram', 'Hansi', 'Hisar', 'Jhajjar', 'Jind', 'Kaithal', 'Karnal', 'Kurukshetra', 'Mahendergarh', 'Mahendragarh', 'Nuh', 'Palwal', 'Panchkula', 'Panipat', 'Rewari', 'Rohtak', 'Sirsa', 'Sonipat', 'Yamunanagar']

FIRMS DAILY

State labels:
['Andaman and Nicobar Islands', 'And

,state,district,district_id
109,Haryana,Ambala,103
126,Haryana,Fatehabad,106
166,Haryana,Gurgaon,107
167,Haryana,Hisar,108
177,Haryana,Jhajjar,109
179,Haryana,Jind,110
199,Haryana,Kaithal,111
227,Haryana,Karnal,112
262,Haryana,Kurukshetra,113
290,Haryana,Palwal,116



FIRMS SEASONAL

State labels:
['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Azad Kashmir', 'Balochistan', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra & Nagar Haveli and Daman & Diu', 'Delhi', 'Gilgit-Baltistan', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Islamabad', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Khyber Pakhtunkhwa', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Sindh', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

Punjab-Haryana records: 10772

District count by state:
state
Haryana    21
Punjab     56
Name: district, dtype: int64

District names:

Punjab (56 districts):
['Amritsar', 'Attock', 'Bahawalnagar', 'Bahawalpur', 'Barnala', 'Bathinda', 'Bhakkar', 'Chakwal', 'Dera Ghazi Khan', 'Faisalabad', 'Faridkot', 'Fatehgarh Sahib', 'Fazilka', 'Firozpur', 'Gujranwala', 'Gujrat'

,state,district,district_id
101,Haryana,Ambala,103
103,Haryana,Fatehabad,106
105,Haryana,Gurgaon,107
106,Haryana,Hisar,108
108,Haryana,Jhajjar,109
109,Haryana,Jind,110
111,Haryana,Kaithal,111
113,Haryana,Karnal,112
115,Haryana,Kurukshetra,113
117,Haryana,Palwal,116



PADDY DISTRICT

State labels:
['Bihar', 'Haryana', 'Himachal Pradesh', 'Madhya Pradesh', 'Punjab', 'Rajasthan', 'Uttar Pradesh', 'Uttarakhand']

Punjab-Haryana records: 1117

District count by state:
state
Haryana    22
Punjab     23
Name: district, dtype: int64

District names:

Punjab (23 districts):
['Amritsar', 'Barnala', 'Bathinda', 'Faridkot', 'Fatehgarh Sahib', 'Fazilka', 'Ferozepur', 'Gurdaspur', 'Hoshiarpur', 'Jalandhar', 'Kapurthala', 'Ludhiana', 'Malerkotla', 'Mansa', 'Moga', 'Pathankot', 'Patiala', 'Rupnagar', 'S.A.S Nagar', 'Sangrur', 'Shahid Bhagat Singh Nagar', 'Sri Muktsar Sahib', 'Tarn Taran']

Haryana (22 districts):
['Ambala', 'Bhiwani', 'Charkhi Dadri', 'Faridabad', 'Fatehabad', 'Gurugram', 'Hisar', 'Jhajjar', 'Jind', 'Kaithal', 'Karnal', 'Kurukshetra', 'Mahendragarh', 'Nuh', 'Palwal', 'Panchkula', 'Panipat', 'Rewari', 'Rohtak', 'Sirsa', 'Sonipat', 'Yamunanagar']

Available geographic identifiers:


,state,district,district_id,lgd_code
1054,Haryana,Ambala,103,58
1082,Haryana,Bhiwani,104,59
1110,Haryana,Charkhi Dadri,104,701
1118,Haryana,Faridabad,105,60
1146,Haryana,Fatehabad,106,61
1174,Haryana,Gurugram,107,62
1202,Haryana,Hisar,108,63
1230,Haryana,Jhajjar,109,64
1258,Haryana,Jind,110,65
1286,Haryana,Kaithal,111,66



BURNED AREA DISTRICT

State labels:
['Chandigarh', 'Delhi', 'Haryana', 'Punjab']

Punjab-Haryana records: 989

District count by state:
state
Haryana    21
Punjab     22
Name: district, dtype: int64

District names:

Punjab (22 districts):
['Amritsar', 'Barnala', 'Bathinda', 'Faridkot', 'Fatehgarh Sahib', 'Fazilka', 'Firozpur', 'Gurdaspur', 'Hoshiarpur', 'Jalandhar', 'Kapurthala', 'Ludhiana', 'Mansa', 'Moga', 'Muktsar', 'Pathankot', 'Patiala', 'Rupnagar', 'Sahibzada Ajit Singh Nagar', 'Sangrur', 'Shahid Bhagat Singh Nagar', 'Tarn Taran']

Haryana (21 districts):
['Ambala', 'Bhiwani', 'Faridabad', 'Fatehabad', 'Gurgaon', 'Hisar', 'Jhajjar', 'Jind', 'Kaithal', 'Karnal', 'Kurukshetra', 'Mahendragarh', 'Mewat', 'Palwal', 'Panchkula', 'Panipat', 'Rewari', 'Rohtak', 'Sirsa', 'Sonipat', 'Yamunanagar']

Available geographic identifiers:


,state,district,district_id
12,Haryana,Ambala,103
13,Haryana,Bhiwani,104
14,Haryana,Faridabad,105
15,Haryana,Fatehabad,106
16,Haryana,Gurgaon,107
17,Haryana,Hisar,108
18,Haryana,Jhajjar,109
19,Haryana,Jind,110
20,Haryana,Kaithal,111
21,Haryana,Karnal,112



DISTRICT BOUNDARIES
Punjab-Haryana boundary records: 77

District count by state:
state
Haryana    21
Punjab     56
Name: name, dtype: int64


,id,name,state
0,103,Ambala,Haryana
1,104,Bhiwani,Haryana
2,105,Faridabad,Haryana
3,106,Fatehabad,Haryana
4,107,Gurgaon,Haryana
...,...,...,...
72,507,Sheikhpura,Punjab
73,508,Sialkot,Punjab
74,281,Tarn Taran,Punjab
75,509,Toba Tek Singh,Punjab



## 🌍 Step 3: Filter Indian Punjab and Haryana Boundaries

The boundary dataset contains multiple geographic regions named Punjab. We must use the country field to isolate India before selecting Punjab and Haryana.

This prevents accidentally including districts from Pakistani Punjab in our project.


In [14]:

# 1. Inspect country values

print("COUNTRY DISTRIBUTION:")
print(district_boundaries["country"].value_counts(dropna=False))

# 2. Inspect Punjab boundary records by country

print("\nPUNJAB BOUNDARIES BY COUNTRY:")
display(
    district_boundaries[
        district_boundaries["name"].notna() &
        district_boundaries["state"].eq("Punjab")
    ].groupby(["country", "state"])
    .size()
    .reset_index(name="district_records")
)

# 3. Filter Indian Punjab and Haryana

ph_boundaries = district_boundaries[
    (district_boundaries["country"].astype("string").str.strip() == "India") &
    (district_boundaries["state"].astype("string").str.strip().isin(
        ["Punjab", "Haryana"]
    ))
].copy()

# 4. Display corrected geography

print("\nINDIAN PUNJAB-HARYANA BOUNDARIES")
print("Total records:", len(ph_boundaries))

print("\nDistrict count by state:")
print(ph_boundaries.groupby("state")["name"].nunique())

print("\nDistrict names:")
display(
    ph_boundaries[["id", "name", "state", "country"]]
    .sort_values(["state", "name"])
    .reset_index(drop=True)
)


COUNTRY DISTRIBUTION:
country
India       735
Pakistan    122
Name: count, dtype: int64

PUNJAB BOUNDARIES BY COUNTRY:


,country,state,district_records
0,India,Punjab,22
1,Pakistan,Punjab,34



INDIAN PUNJAB-HARYANA BOUNDARIES
Total records: 43

District count by state:
state
Haryana    21
Punjab     22
Name: name, dtype: int64

District names:


,id,name,state,country
0,103,Ambala,Haryana,India
1,104,Bhiwani,Haryana,India
2,105,Faridabad,Haryana,India
3,106,Fatehabad,Haryana,India
4,107,Gurgaon,Haryana,India
5,108,Hisar,Haryana,India
6,109,Jhajjar,Haryana,India
7,110,Jind,Haryana,India
8,111,Kaithal,Haryana,India
9,112,Karnal,Haryana,India



## 🔍 Step 4: Identify District Name Mismatches

### Objective
Compare district names in each selected dataset against the Indian Punjab-Haryana boundary reference.

We will identify exact matches and unmatched names after basic text normalization. No district will be automatically renamed or discarded.

This comparison will help us create a reliable district mapping table.


In [15]:

import re

# Normalize names only for comparison
def normalize_name(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9\s]", "", value)
    value = re.sub(r"\s+", " ", value)

    return value


# Reference boundary keys
boundary_reference = ph_boundaries[["state", "name", "id"]].copy()

boundary_reference["state_key"] = (
    boundary_reference["state"].apply(normalize_name)
)

boundary_reference["district_key"] = (
    boundary_reference["name"].apply(normalize_name)
)

boundary_keys = set(
    zip(
        boundary_reference["state_key"],
        boundary_reference["district_key"]
    )
)


# Compare each dataset with boundary reference
def compare_district_names(dataset_name, df):

    temp = df.copy()

    # Keep only Punjab and Haryana
    temp = temp[
        temp["state"].astype("string").str.strip().isin(
            ["Punjab", "Haryana"]
        )
    ].copy()

    # Exclude non-India records wherever country is available
    if "country" in temp.columns:
        temp = temp[
            temp["country"].astype("string").str.strip() == "India"
        ]

    temp["state_key"] = temp["state"].apply(normalize_name)
    temp["district_key"] = temp["district"].apply(normalize_name)

    unique_districts = temp[
        ["state", "district", "state_key", "district_key"]
    ].drop_duplicates()

    unique_districts["boundary_match"] = [
        (state, district) in boundary_keys
        for state, district in zip(
            unique_districts["state_key"],
            unique_districts["district_key"]
        )
    ]

    print("\n" + "=" * 65)
    print(dataset_name.upper())
    print("=" * 65)

    print("Unique district labels:", len(unique_districts))
    print(
        "Matched labels:",
        unique_districts["boundary_match"].sum()
    )
    print(
        "Unmatched labels:",
        (~unique_districts["boundary_match"]).sum()
    )

    print("\nUnmatched districts:")
    display(
        unique_districts.loc[
            ~unique_districts["boundary_match"],
            ["state", "district"]
        ].sort_values(["state", "district"])
        .reset_index(drop=True)
    )


# Run comparisons
compare_district_names("CREAMS Daily", creams_daily)
compare_district_names("FIRMS Daily", fires_daily)
compare_district_names("FIRMS Seasonal", fires_season)
compare_district_names("Paddy District", paddy_district)
compare_district_names("Burned Area District", burned_district)



CREAMS DAILY
Unique district labels: 51
Matched labels: 38
Unmatched labels: 13

Unmatched districts:


,state,district
0,Haryana,Charkhi Dadri
1,Haryana,Faridkot
2,Haryana,Faridkot Faridabad
3,Haryana,Gurugram
4,Haryana,Hansi
5,Haryana,Mahendergarh
6,Haryana,Nuh
7,Punjab,Kharar
8,Punjab,Malerkotla
9,Punjab,Punjab



FIRMS DAILY
Unique district labels: 43
Matched labels: 43
Unmatched labels: 0

Unmatched districts:


,state,district



FIRMS SEASONAL
Unique district labels: 43
Matched labels: 43
Unmatched labels: 0

Unmatched districts:


,state,district



PADDY DISTRICT
Unique district labels: 45
Matched labels: 38
Unmatched labels: 7

Unmatched districts:


,state,district
0,Haryana,Charkhi Dadri
1,Haryana,Gurugram
2,Haryana,Nuh
3,Punjab,Ferozepur
4,Punjab,Malerkotla
5,Punjab,S.A.S Nagar
6,Punjab,Sri Muktsar Sahib



BURNED AREA DISTRICT
Unique district labels: 43
Matched labels: 43
Unmatched labels: 0

Unmatched districts:


,state,district



## 🗺️ Step 5: Create Verified District Name Aliases

Create a controlled mapping for district names that refer to the same geographical district under different naming conventions.

Only confirmed aliases will be standardized. Newer districts and suspicious labels will remain unresolved until their geographical boundaries are verified.


In [16]:

# Confirmed aliases: (state, source_name) -> boundary_name

district_aliases = {
    ("Haryana", "Gurugram"): "Gurgaon",
    ("Haryana", "Mahendergarh"): "Mahendragarh",
    ("Haryana", "Nuh"): "Mewat",

    ("Punjab", "SAS Nagar"): "Sahibzada Ajit Singh Nagar",
    ("Punjab", "SBS Nagar"): "Shahid Bhagat Singh Nagar",
    ("Punjab", "Sri Muktsar Sahib"): "Muktsar",
    ("Punjab", "Ferozepur"): "Firozpur",
    ("Punjab", "S.A.S Nagar"): "Sahibzada Ajit Singh Nagar"
}

# Create a reusable mapping function

def standardize_district(state, district):
    if pd.isna(state) or pd.isna(district):
        return district

    return district_aliases.get(
        (str(state).strip(), str(district).strip()),
        str(district).strip()
    )


print("CONFIRMED DISTRICT ALIASES:")
for key, value in district_aliases.items():
    print(f"{key[0]}: {key[1]} -> {value}")

print("\nTotal confirmed aliases:", len(district_aliases))


CONFIRMED DISTRICT ALIASES:
Haryana: Gurugram -> Gurgaon
Haryana: Mahendergarh -> Mahendragarh
Haryana: Nuh -> Mewat
Punjab: SAS Nagar -> Sahibzada Ajit Singh Nagar
Punjab: SBS Nagar -> Shahid Bhagat Singh Nagar
Punjab: Sri Muktsar Sahib -> Muktsar
Punjab: Ferozepur -> Firozpur
Punjab: S.A.S Nagar -> Sahibzada Ajit Singh Nagar

Total confirmed aliases: 8



## 🔢 Step 6: Verify District IDs Against Geographic Boundaries

Compare the district IDs in FIRMS, Paddy and Burned Area datasets with the IDs in our India-only Punjab and Haryana boundary reference.

We will check:
- Whether IDs exist in the boundary reference
- Whether the state matches
- Whether district names match after standardization
- Which records require further investigation

**Important:** An ID match alone is not sufficient. Its state and district identity must also be consistent.


In [17]:

def audit_district_ids(dataset_name, df):

    print(f"\n{'=' * 25} {dataset_name.upper()} {'=' * 25}")

    temp = df.copy()

    temp = temp[
        temp["state"].astype("string").str.strip().isin(
            ["Punjab", "Haryana"]
        )
    ].copy()

    if "country" in temp.columns:
        temp = temp[
            temp["country"].astype("string").str.strip() == "India"
        ].copy()

    temp["district_id_numeric"] = pd.to_numeric(
        temp["district_id"], errors="coerce"
    )

    unique_records = temp[
        ["state", "district", "district_id_numeric"]
    ].drop_duplicates().copy()

    result = unique_records.merge(
        boundary_id_reference[
            ["id", "state", "name"]
        ],
        left_on="district_id_numeric",
        right_on="id",
        how="left",
        suffixes=("_source", "_boundary")
    )

    result["state_match"] = (
        result["state_source"].apply(normalize_name)
        == result["state_boundary"].apply(normalize_name)
    )

    result["name_match"] = (
        result["district"].apply(normalize_name)
        == result["name"].apply(normalize_name)
    )

    result["valid_id_match"] = (
        result["id"].notna()
        & result["state_match"]
    )

    print("Unique source district-ID combinations:", len(result))
    print("IDs found in boundary reference:", result["id"].notna().sum())
    print("IDs with matching state:", result["valid_id_match"].sum())
    print("IDs with matching district name:", result["name_match"].sum())

    print("\nUnmatched or conflicting records:")

    display(
        result[
            ~result["valid_id_match"] | ~result["name_match"]
        ][
            [
                "state_source",
                "district",
                "district_id_numeric",
                "name",
                "state_boundary",
                "valid_id_match",
                "name_match"
            ]
        ].drop_duplicates()
    )


audit_district_ids("FIRMS Daily", fires_daily)
audit_district_ids("FIRMS Seasonal", fires_season)
audit_district_ids("Paddy District", paddy_district)
audit_district_ids("Burned Area District", burned_district)



========================= FIRMS DAILY =========================


NameError: name 'boundary_id_reference' is not defined


## 🔍 Step 7: Investigate Paddy District ID Mismatches

Inspect the seven Paddy district records whose names differ from their matched boundary districts.

Compare their district IDs, LGD codes and year coverage before deciding whether they can safely be integrated.


In [ ]:

paddy_mismatches = [
    ("Haryana", "Charkhi Dadri"),
    ("Haryana", "Gurugram"),
    ("Haryana", "Nuh"),
    ("Punjab", "Ferozepur"),
    ("Punjab", "Malerkotla"),
    ("Punjab", "S.A.S Nagar"),
    ("Punjab", "Sri Muktsar Sahib")
]

paddy_review = paddy_district[
    paddy_district.apply(
        lambda row: (row["state"], row["district"]) in paddy_mismatches,
        axis=1
    )
].copy()

# Show original records without modifying anything
display(
    paddy_review[
        [
            "state",
            "district",
            "lgd_code",
            "district_id",
            "year",
            "kharif_area_ha",
            "total_area_ha"
        ]
    ].sort_values(["state", "district", "year"])
)

print("\nDISTRICT-LEVEL ID SUMMARY:")

display(
    paddy_review.groupby(
        ["state", "district"],
        dropna=False
    ).agg(
        district_ids=("district_id", lambda x: sorted(x.dropna().unique().tolist())),
        lgd_codes=("lgd_code", lambda x: sorted(x.dropna().unique().tolist())),
        first_year=("year", "min"),
        last_year=("year", "max"),
        records=("year", "size")
    ).reset_index()
)


,state,district,lgd_code,district_id,year,kharif_area_ha,total_area_ha
1110,Haryana,Charkhi Dadri,701,104,2017,9400.0,9400
1111,Haryana,Charkhi Dadri,701,104,2018,10000.0,10000
1112,Haryana,Charkhi Dadri,701,104,2019,12000.0,12000
1113,Haryana,Charkhi Dadri,701,104,2020,5300.0,5300
1114,Haryana,Charkhi Dadri,701,104,2021,4879.0,4879
...,...,...,...,...,...,...,...
3675,Punjab,Sri Muktsar Sahib,39,274,2020,185400.0,185400
3676,Punjab,Sri Muktsar Sahib,39,274,2021,183400.0,183400
3677,Punjab,Sri Muktsar Sahib,39,274,2022,183000.0,183000
3678,Punjab,Sri Muktsar Sahib,39,274,2023,187500.0,187500



DISTRICT-LEVEL ID SUMMARY:


,state,district,district_ids,lgd_codes,first_year,last_year,records
0,Haryana,Charkhi Dadri,[104],[701],2017,2024,8
1,Haryana,Gurugram,[107],[62],1997,2024,28
2,Haryana,Nuh,[115],[604],2005,2024,20
3,Punjab,Ferozepur,[266],[31],1997,2024,28
4,Punjab,Malerkotla,[279],[737],2021,2024,4
5,Punjab,S.A.S Nagar,[278],[608],2006,2024,19
6,Punjab,Sri Muktsar Sahib,[274],[39],1997,2024,28



## 🧾 Step 8: Validate Paddy LGD Codes

Check whether each LGD code consistently identifies one district and one state across all years.

We will also identify missing codes and any codes associated with multiple district labels.


In [ ]:

# Keep records with available LGD codes
paddy_lgd = paddy_district[
    paddy_district["lgd_code"].notna()
].copy()

# Check one LGD code against multiple district/state labels
lgd_consistency = (
    paddy_lgd.groupby("lgd_code")
    .agg(
        unique_districts=("district", "nunique"),
        unique_states=("state", "nunique"),
        districts=("district", lambda x: sorted(x.unique())),
        states=("state", lambda x: sorted(x.unique())),
        records=("year", "size"),
        first_year=("year", "min"),
        last_year=("year", "max")
    )
    .reset_index()
)

print("TOTAL PADDY RECORDS:", len(paddy_district))
print("MISSING LGD CODES:", paddy_district["lgd_code"].isna().sum())
print("UNIQUE LGD CODES:", paddy_lgd["lgd_code"].nunique())

print("\nLGD CODES WITH MULTIPLE DISTRICTS OR STATES:")

display(
    lgd_consistency[
        (lgd_consistency["unique_districts"] > 1)
        | (lgd_consistency["unique_states"] > 1)
    ]
)

print("\nSEVEN PREVIOUSLY IDENTIFIED DISTRICTS:")

display(
    lgd_consistency[
        lgd_consistency["districts"].apply(
            lambda names: any(
                name in names
                for name in [
                    "Charkhi Dadri", "Gurugram", "Nuh",
                    "Ferozepur", "Malerkotla",
                    "S.A.S Nagar", "Sri Muktsar Sahib"
                ]
            )
        )
    ]
)


TOTAL PADDY RECORDS: 6674
MISSING LGD CODES: 0
UNIQUE LGD CODES: 267

LGD CODES WITH MULTIPLE DISTRICTS OR STATES:


,lgd_code,unique_districts,unique_states,districts,states,records,first_year,last_year



SEVEN PREVIOUSLY IDENTIFIED DISTRICTS:


,lgd_code,unique_districts,unique_states,districts,states,records,first_year,last_year
15,31,1,1,[Ferozepur],[Punjab],28,1997,2024
23,39,1,1,[Sri Muktsar Sahib],[Punjab],28,1997,2024
45,62,1,1,[Gurugram],[Haryana],28,1997,2024
241,604,1,1,[Nuh],[Haryana],20,2005,2024
243,608,1,1,[S.A.S Nagar],[Punjab],19,2006,2024
258,701,1,1,[Charkhi Dadri],[Haryana],8,2017,2024
260,737,1,1,[Malerkotla],[Punjab],4,2021,2024



## 🔗 Step 9: Prepare Datasets for Geographic Integration

Create integration copies with standardized district identifiers.

Preserve original identifiers and district names. Do not force unmatched districts into existing boundary records.

This step prepares the datasets for safe integration without losing historical or administrative information.


In [ ]:

# 1. Create separate integration copies

creams_integrated = creams_daily.copy()
fires_daily_integrated = fires_daily.copy()
fires_season_integrated = fires_season.copy()
paddy_integrated = paddy_district.copy()
burned_integrated = burned_district.copy()


# 2. FIRMS: verified boundary IDs

fires_daily_integrated["boundary_id"] = pd.to_numeric(
    fires_daily_integrated["district_id"], errors="coerce"
)

fires_season_integrated["boundary_id"] = pd.to_numeric(
    fires_season_integrated["district_id"], errors="coerce"
)


# 3. Burned Area: verified boundary IDs

burned_integrated["boundary_id"] = pd.to_numeric(
    burned_integrated["district_id"], errors="coerce"
)


# 4. Paddy: preserve LGD codes separately

paddy_integrated["lgd_id"] = pd.to_numeric(
    paddy_integrated["lgd_code"], errors="coerce"
)

# Standardize confirmed district aliases for name-based matching
paddy_integrated["district_standardized"] = (
    paddy_integrated.apply(
        lambda row: standardize_district(
            row["state"], row["district"]
        ),
        axis=1
    )
)


# 5. CREAMS: standardize confirmed aliases

creams_integrated["district_standardized"] = (
    creams_integrated.apply(
        lambda row: standardize_district(
            row["state"], row["district"]
        ),
        axis=1
    )
)


# 6. Preserve unresolved CREAMS labels for review

creams_integrated["geography_review_required"] = (
    creams_integrated["district_standardized"].isin(
        [
            "Charkhi Dadri",
            "Faridkot",
            "Faridkot Faridabad",
            "Hansi",
            "Kharar",
            "Malerkotla",
            "Punjab"
        ]
    )
)


# 7. Print integration summary

print("INTEGRATION COPIES CREATED")

for name, df in {
    "CREAMS": creams_integrated,
    "FIRMS Daily": fires_daily_integrated,
    "FIRMS Seasonal": fires_season_integrated,
    "Paddy": paddy_integrated,
    "Burned Area": burned_integrated
}.items():
    print(f"{name}: {df.shape}")

print("\nPADDY LGD IDs missing:",
      paddy_integrated["lgd_id"].isna().sum())

print("FIRMS Daily boundary IDs missing:",
      fires_daily_integrated["boundary_id"].isna().sum())

print("FIRMS Seasonal boundary IDs missing:",
      fires_season_integrated["boundary_id"].isna().sum())

print("Burned Area boundary IDs missing:",
      burned_integrated["boundary_id"].isna().sum())

print("\nCREAMS records requiring geographic review:",
      creams_integrated["geography_review_required"].sum())


INTEGRATION COPIES CREATED
CREAMS: (198423, 15)
FIRMS Daily: (427728, 9)
FIRMS Seasonal: (84875, 9)
Paddy: (6674, 14)
Burned Area: (1265, 7)

PADDY LGD IDs missing: 0
FIRMS Daily boundary IDs missing: 0
FIRMS Seasonal boundary IDs missing: 0
Burned Area boundary IDs missing: 0

CREAMS records requiring geographic review: 3211



## 🧭 Step 10: Review Unresolved CREAMS Geographic Records

Analyze CREAMS records flagged for geographic review.

Identify:
- Unresolved district labels
- State-wise record counts
- Year-wise distribution
- Whether these records contain usable daily fire observations

Do not delete or modify any records during this review.


In [ ]:

creams_review = creams_integrated[
    creams_integrated["geography_review_required"]
].copy()

print("TOTAL RECORDS REQUIRING REVIEW:", len(creams_review))

print("\n1. DISTRICT-WISE SUMMARY")

display(
    creams_review.groupby(
        ["state", "district_standardized"],
        dropna=False
    ).agg(
        total_records=("date", "size"),
        daily_values_available=("fires_day", "count"),
        first_date=("date", "min"),
        last_date=("date", "max")
    ).reset_index().sort_values(
        ["state", "total_records"],
        ascending=[True, False]
    )
)

print("\n2. YEAR-WISE DISTRIBUTION")

creams_review["observation_year"] = creams_review["date"].dt.year

display(
    creams_review.groupby(
        ["state", "district_standardized", "observation_year"],
        dropna=False
    ).size().reset_index(name="records")
)

print("\n3. DAILY TARGET AVAILABILITY")

display(
    creams_review.groupby(
        ["state", "district_standardized"],
        dropna=False
    )["fires_day"].agg(
        total_records="size",
        available="count"
    ).reset_index()
)


TOTAL RECORDS REQUIRING REVIEW: 3211

1. DISTRICT-WISE SUMMARY


,state,district_standardized,total_records,daily_values_available,first_date,last_date
0,Haryana,Charkhi Dadri,1042,37,2016-10-01,2026-10-27
3,Haryana,Hansi,301,1,2020-09-15,2026-10-27
1,Haryana,Faridkot,5,0,2016-10-02,2020-10-02
2,Haryana,Faridkot Faridabad,5,0,2016-10-04,2020-10-04
4,Punjab,Faridkot,1046,430,2016-10-01,2026-10-27
6,Punjab,Malerkotla,810,119,2020-09-15,2026-10-27
5,Punjab,Kharar,1,1,2021-10-20,2021-10-20
7,Punjab,Punjab,1,1,2026-05-12,2026-05-12



2. YEAR-WISE DISTRIBUTION


,state,district_standardized,observation_year,records
0,Haryana,Charkhi Dadri,2016,58
1,Haryana,Charkhi Dadri,2017,58
2,Haryana,Charkhi Dadri,2018,58
3,Haryana,Charkhi Dadri,2019,58
4,Haryana,Charkhi Dadri,2020,77
5,Haryana,Charkhi Dadri,2021,77
6,Haryana,Charkhi Dadri,2022,138
7,Haryana,Charkhi Dadri,2023,138
8,Haryana,Charkhi Dadri,2024,138
9,Haryana,Charkhi Dadri,2025,138



3. DAILY TARGET AVAILABILITY


,state,district_standardized,total_records,available
0,Haryana,Charkhi Dadri,1042,37
1,Haryana,Faridkot,5,0
2,Haryana,Faridkot Faridabad,5,0
3,Haryana,Hansi,301,1
4,Punjab,Faridkot,1046,430
5,Punjab,Kharar,1,1
6,Punjab,Malerkotla,810,119
7,Punjab,Punjab,1,1



## 🗺️ Step 11: Classify CREAMS Geographic Records

Assign each CREAMS record a geographic status based on its state and standardized district name.

- Boundary match: valid district found in our current boundary reference.
- Valid district without boundary: newer administrative district absent from the current boundary file.
- Suspicious label: district label requires source verification.

Preserve all original records. This classification does not delete or alter observations.


In [ ]:

# Newer districts missing from our current boundary reference
new_districts = {
    ("Haryana", "Charkhi Dadri"),
    ("Haryana", "Hansi"),
    ("Punjab", "Malerkotla")
}

# Suspicious labels requiring source verification
suspicious_districts = {
    ("Haryana", "Faridkot"),
    ("Haryana", "Faridkot Faridabad"),
    ("Punjab", "Kharar"),
    ("Punjab", "Punjab")
}


def classify_geography(row):

    state = str(row["state"]).strip()
    district = str(row["district_standardized"]).strip()

    key = (state, district)

    if key in suspicious_districts:
        return "suspicious_label"

    if key in new_districts:
        return "valid_district_no_boundary"

    normalized_key = (
        normalize_name(state),
        normalize_name(district)
    )

    if normalized_key in boundary_keys:
        return "boundary_match"

    return "unresolved"


# Apply classification
creams_integrated["geography_status"] = (
    creams_integrated.apply(classify_geography, axis=1)
)


# Overall summary
print("GEOGRAPHIC STATUS SUMMARY:")

display(
    creams_integrated["geography_status"]
    .value_counts(dropna=False)
    .rename_axis("geography_status")
    .reset_index(name="records")
)


# District-level summary
print("\nDISTRICT-LEVEL GEOGRAPHIC CLASSIFICATION:")

display(
    creams_integrated.groupby(
        ["state", "district_standardized", "geography_status"],
        dropna=False
    ).size().reset_index(name="records")
    .sort_values(["geography_status", "state", "district_standardized"])
)


# Verify Punjab Faridkot classification
print("\nPUNJAB FARIDKOT CHECK:")

display(
    creams_integrated[
        (creams_integrated["state"] == "Punjab")
        & (creams_integrated["district_standardized"] == "Faridkot")
    ]["geography_status"].value_counts()
)


GEOGRAPHIC STATUS SUMMARY:


,geography_status,records
0,unresolved,150756
1,boundary_match,45502
2,valid_district_no_boundary,2153
3,suspicious_label,12



DISTRICT-LEVEL GEOGRAPHIC CLASSIFICATION:


,state,district_standardized,geography_status,records
11,Haryana,Ambala,boundary_match,1054
12,Haryana,Bhiwani,boundary_match,1054
14,Haryana,Faridabad,boundary_match,1047
17,Haryana,Fatehabad,boundary_match,1054
18,Haryana,Gurgaon,boundary_match,1054
...,...,...,...,...
247,Uttarakhand,Nainital,unresolved,301
248,Uttarakhand,Udham Singh Nagar,unresolved,301
13,Haryana,Charkhi Dadri,valid_district_no_boundary,1042
19,Haryana,Hansi,valid_district_no_boundary,301



PUNJAB FARIDKOT CHECK:


geography_status
boundary_match    1046
Name: count, dtype: int64


## 🌍 Step 12: Separate Project Geography From Outside-Scope Records

The project focuses exclusively on Punjab and Haryana.

Classify records from other states as outside project scope instead of unresolved. Within Punjab and Haryana, retain the distinction between matched districts, newer districts without boundaries, suspicious labels and genuinely unresolved districts.


In [ ]:

def classify_geography(row):

    state = str(row["state"]).strip()
    district = str(row["district_standardized"]).strip()

    # Our project covers only Punjab and Haryana
    if state not in ["Punjab", "Haryana"]:
        return "outside_project_scope"

    key = (state, district)

    if key in suspicious_districts:
        return "suspicious_label"

    if key in new_districts:
        return "valid_district_no_boundary"

    normalized_key = (
        normalize_name(state),
        normalize_name(district)
    )

    if normalized_key in boundary_keys:
        return "boundary_match"

    return "unresolved_in_scope"


# Reclassify all records
creams_integrated["geography_status"] = (
    creams_integrated.apply(classify_geography, axis=1)
)

print("CORRECTED GEOGRAPHIC STATUS SUMMARY:")

display(
    creams_integrated["geography_status"]
    .value_counts()
    .rename_axis("geography_status")
    .reset_index(name="records")
)

print("\nPUNJAB-HARYANA UNRESOLVED DISTRICTS:")

display(
    creams_integrated[
        creams_integrated["geography_status"] == "unresolved_in_scope"
    ].groupby(
        ["state", "district_standardized"]
    ).size().reset_index(name="records")
)

print("\nPUNJAB-HARYANA TOTAL:")

print(
    creams_integrated[
        creams_integrated["state"].isin(["Punjab", "Haryana"])
    ].shape[0]
)


CORRECTED GEOGRAPHIC STATUS SUMMARY:


,geography_status,records
0,outside_project_scope,150756
1,boundary_match,45502
2,valid_district_no_boundary,2153
3,suspicious_label,12



PUNJAB-HARYANA UNRESOLVED DISTRICTS:


,state,district_standardized,records



PUNJAB-HARYANA TOTAL:
47667



## 🌾 Step 13: Create the Punjab-Haryana CREAMS Dataset

Filter CREAMS to Punjab and Haryana.

Retain:
- Districts matching the existing boundary reference.
- Valid newer districts without available boundary polygons.

Exclude suspicious geographic labels from this project-specific subset, while preserving them in the original integration dataset for auditability.




In [ ]:

# Create Punjab-Haryana project dataset
creams_ph = creams_integrated[
    creams_integrated["state"].isin(["Punjab", "Haryana"])
].copy()

# Retain geographically valid records
creams_ph = creams_ph[
    creams_ph["geography_status"].isin(
        ["boundary_match", "valid_district_no_boundary"]
    )
].copy()

print("CREAMS PUNJAB-HARYANA SHAPE:", creams_ph.shape)

print("\nSTATE DISTRIBUTION:")
display(creams_ph["state"].value_counts())

print("\nGEOGRAPHIC STATUS:")
display(creams_ph["geography_status"].value_counts())

print("\nDISTRICT COVERAGE:")
display(
    creams_ph.groupby(
        ["state", "district_standardized"]
    ).agg(
        total_records=("date", "size"),
        daily_target_available=("fires_day", "count"),
        first_date=("date", "min"),
        last_date=("date", "max")
    ).reset_index().sort_values(["state", "district_standardized"])
)

print("\nDAILY TARGET COVERAGE:")
print("Available:", creams_ph["fires_day"].notna().sum())
print("Missing:", creams_ph["fires_day"].isna().sum())
print("Total:", len(creams_ph))


CREAMS PUNJAB-HARYANA SHAPE: (47655, 16)

STATE DISTRIBUTION:


state
Punjab     23927
Haryana    23728
Name: count, dtype: int64


GEOGRAPHIC STATUS:


geography_status
boundary_match                45502
valid_district_no_boundary     2153
Name: count, dtype: int64


DISTRICT COVERAGE:


,state,district_standardized,total_records,daily_target_available,first_date,last_date
0,Haryana,Ambala,1054,358,2016-10-01,2026-10-27
1,Haryana,Bhiwani,1054,130,2016-10-01,2026-10-27
2,Haryana,Charkhi Dadri,1042,37,2016-10-01,2026-10-27
3,Haryana,Faridabad,1047,133,2016-10-01,2026-10-27
4,Haryana,Fatehabad,1054,453,2016-10-01,2026-10-27
5,Haryana,Gurgaon,1054,32,2016-10-01,2026-10-27
6,Haryana,Hansi,301,1,2020-09-15,2026-10-27
7,Haryana,Hisar,1054,364,2016-10-01,2026-10-27
8,Haryana,Jhajjar,1054,160,2016-10-01,2026-10-27
9,Haryana,Jind,1054,452,2016-10-01,2026-10-27



DAILY TARGET COVERAGE:
Available: 14392
Missing: 33263
Total: 47655



## 🔍 Step 14: Check Duplicate Records After District Standardization

Check whether different original district labels have been converted into the same standardized district and date.

Identify duplicate district-date-season combinations and inspect their original names and target values.

Do not remove duplicates until their source and meaning are understood.


In [ ]:

# 1. Inspect original labels contributing to each standardized district

print("ORIGINAL LABELS AFTER STANDARDIZATION:")

display(
    creams_ph.groupby(
        ["state", "district_standardized", "district"]
    ).size().reset_index(name="records")
    .sort_values(["state", "district_standardized", "district"])
)


# 2. Check duplicate district-date-season combinations

duplicate_keys = [
    "state",
    "district_standardized",
    "date",
    "season",
    "season_year"
]

duplicate_mask = creams_ph.duplicated(
    subset=duplicate_keys,
    keep=False
)

creams_duplicates = creams_ph[duplicate_mask].copy()

print("\nDUPLICATE RECORDS:", len(creams_duplicates))
print("DUPLICATE GROUPS:",
      creams_duplicates.groupby(duplicate_keys).ngroups)

# 3. Inspect duplicate records

display(
    creams_duplicates[
        [
            "state",
            "district",
            "district_standardized",
            "date",
            "season",
            "season_year",
            "fires_day",
            "fires_cumulative",
            "day_source"
        ]
    ].sort_values(
        ["state", "district_standardized", "date"]
    ).head(50)
)


# 4. Check specifically for Mahendragarh

print("\nMAHENDRAGARH ORIGINAL LABEL SUMMARY:")

display(
    creams_ph[
        (creams_ph["state"] == "Haryana")
        & (creams_ph["district_standardized"] == "Mahendragarh")
    ].groupby("district").agg(
        records=("date", "size"),
        available_daily_counts=("fires_day", "count"),
        first_date=("date", "min"),
        last_date=("date", "max")
    ).reset_index()
)


ORIGINAL LABELS AFTER STANDARDIZATION:


,state,district_standardized,district,records
0,Haryana,Ambala,Ambala,1054
1,Haryana,Bhiwani,Bhiwani,1054
2,Haryana,Charkhi Dadri,Charkhi Dadri,1042
3,Haryana,Faridabad,Faridabad,1047
4,Haryana,Fatehabad,Fatehabad,1054
5,Haryana,Gurgaon,Gurugram,1054
6,Haryana,Hansi,Hansi,301
7,Haryana,Hisar,Hisar,1054
8,Haryana,Jhajjar,Jhajjar,1054
9,Haryana,Jind,Jind,1054



DUPLICATE RECORDS: 516
DUPLICATE GROUPS: 258


,state,district,district_standardized,date,season,season_year,fires_day,fires_cumulative,day_source
28988,Haryana,Mahendergarh,Mahendragarh,2020-09-15,rice,2020,NaN,0.0,NaN
28989,Haryana,Mahendragarh,Mahendragarh,2020-09-15,rice,2020,NaN,0.0,NaN
29222,Haryana,Mahendergarh,Mahendragarh,2020-09-16,rice,2020,NaN,0.0,NaN
29223,Haryana,Mahendragarh,Mahendragarh,2020-09-16,rice,2020,NaN,0.0,NaN
29456,Haryana,Mahendergarh,Mahendragarh,2020-09-17,rice,2020,NaN,0.0,NaN
29457,Haryana,Mahendragarh,Mahendragarh,2020-09-17,rice,2020,NaN,0.0,NaN
29690,Haryana,Mahendergarh,Mahendragarh,2020-09-18,rice,2020,NaN,0.0,NaN
29691,Haryana,Mahendragarh,Mahendragarh,2020-09-18,rice,2020,NaN,0.0,NaN
29924,Haryana,Mahendergarh,Mahendragarh,2020-09-19,rice,2020,NaN,0.0,NaN
29925,Haryana,Mahendragarh,Mahendragarh,2020-09-19,rice,2020,NaN,0.0,NaN



MAHENDRAGARH ORIGINAL LABEL SUMMARY:


,district,records,available_daily_counts,first_date,last_date
0,Mahendergarh,301,0,2020-09-15,2026-10-27
1,Mahendragarh,1011,4,2016-10-01,2026-05-31


### 🔍 Mahendragarh Duplicate Record Verification

We are checking whether the two spellings represent duplicate copies
of the same records or contain different information.

We will compare all columns except the original and standardized
district names.

⚠️ No records will be deleted during this audit.

In [ ]:


# Columns to compare: exclude only the two district-name columns
comparison_columns = [
    col for col in creams_duplicates.columns
    if col not in ["district", "district_standardized"]
]

# Compare all remaining columns within each duplicate group
comparison = (
    creams_duplicates
    .groupby(duplicate_keys, dropna=False)[comparison_columns]
    .nunique(dropna=False)
)

# Identify groups where at least one compared column differs
different_groups = comparison[
    comparison.gt(1).any(axis=1)
]

print("TOTAL DUPLICATE GROUPS:", len(comparison))
print("GROUPS WITH DIFFERENT VALUES:", len(different_groups))
print("GROUPS WITH IDENTICAL VALUES:", len(comparison) - len(different_groups))

# Display the actual differing records
if not different_groups.empty:

    different_keys = different_groups.index.to_frame(index=False)

    different_records = creams_duplicates.merge(
        different_keys,
        on=duplicate_keys,
        how="inner"
    )

    display(
        different_records.sort_values(
            ["date", "district"]
        )
    )

else:
    print("All duplicate groups are identical across compared columns.")


TOTAL DUPLICATE GROUPS: 258
GROUPS WITH DIFFERENT VALUES: 258
GROUPS WITH IDENTICAL VALUES: 0


,date,season,season_year,state,district,fires_day,fires_cumulative,cum_period_start,day_source,cum_source,cum_check,bulletin_file,cum_bulletin_file,district_standardized,geography_review_required,geography_status
0,2020-09-15,rice,2020,Haryana,Mahendergarh,NaN,0.0,2020-09-15,NaN,district_table,sum_ok,01.RiceResidueFireBulletin_15Sep_2026_ICAR.pdf,01.RiceResidueFireBulletin_15Sep_2026_ICAR.pdf,Mahendragarh,False,boundary_match
1,2020-09-15,rice,2020,Haryana,Mahendragarh,NaN,0.0,2020-09-15,NaN,district_table,sum_ok,01.RiceResidueFireBulletin_15Sep_2025_ICAR.pdf,01.RiceResidueFireBulletin_15Sep_2025_ICAR.pdf,Mahendragarh,False,boundary_match
2,2020-09-16,rice,2020,Haryana,Mahendergarh,NaN,0.0,2020-09-15,NaN,district_table,sum_ok,02.RiceResidueFireBulletin_16Sep_2026_ICAR.pdf,02.RiceResidueFireBulletin_16Sep_2026_ICAR.pdf,Mahendragarh,False,boundary_match
3,2020-09-16,rice,2020,Haryana,Mahendragarh,NaN,0.0,2020-09-15,NaN,district_table,sum_ok,02.RiceResidueFireBulletin_16Sep_2025_ICAR.pdf,02.RiceResidueFireBulletin_16Sep_2025_ICAR.pdf,Mahendragarh,False,boundary_match
4,2020-09-17,rice,2020,Haryana,Mahendergarh,NaN,0.0,2020-09-15,NaN,district_table,sum_ok,03.RiceResidueFireBulletin_17Sep_2026_ICAR.pdf,03.RiceResidueFireBulletin_17Sep_2026_ICAR.pdf,Mahendragarh,False,boundary_match
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511,2025-10-25,rice,2025,Haryana,Mahendragarh,NaN,0.0,2025-09-15,NaN,district_table,sum_ok,41.RiceResidueFireBulletin_25Oct_2025_ICAR.pdf,41.RiceResidueFireBulletin_25Oct_2025_ICAR.pdf,Mahendragarh,False,boundary_match
512,2025-10-26,rice,2025,Haryana,Mahendergarh,NaN,0.0,2025-09-15,NaN,district_table,sum_ok,42.RiceResidueFireBulletin_26Oct_2026_ICAR.pdf,42.RiceResidueFireBulletin_26Oct_2026_ICAR.pdf,Mahendragarh,False,boundary_match
513,2025-10-26,rice,2025,Haryana,Mahendragarh,NaN,0.0,2025-09-15,NaN,district_table,sum_ok,42.RiceResidueFireBulletin_26Oct_2025_ICAR.pdf,42.RiceResidueFireBulletin_26Oct_2025_ICAR.pdf,Mahendragarh,False,boundary_match
514,2025-10-27,rice,2025,Haryana,Mahendergarh,NaN,0.0,2025-09-15,NaN,district_table,sum_ok,43.RiceResidueFireBulletin_27Oct_2026_ICAR.pdf,43.RiceResidueFireBulletin_27Oct_2026_ICAR.pdf,Mahendragarh,False,boundary_match


### 📊 Column-wise Difference Analysis — Mahendragarh

We identified differences in all 258 duplicate groups.

Now we will calculate how many groups differ in each column.
This will help us distinguish meaningful source discrepancies
from differences in metadata or missing values.

⚠️ Original records remain unchanged.

In [ ]:
# Count how many duplicate groups differ in each column

column_difference_counts = comparison.gt(1).sum().sort_values(
    ascending=False
)

column_difference_summary = (
    column_difference_counts
    .reset_index()
)

column_difference_summary.columns = [
    "column",
    "groups_with_differences"
]

display(column_difference_summary)

# Inspect the two original district labels separately
print("\nMAHENDERGARH SUMMARY:")

display(
    creams_duplicates.groupby("district").agg(
        records=("date", "size"),
        available_daily_counts=("fires_day", "count"),
        available_cumulative_counts=("fires_cumulative", "count"),
        available_day_sources=("day_source", "count"),
        available_cum_sources=("cum_source", "count"),
        first_date=("date", "min"),
        last_date=("date", "max")
    )
)

,column,groups_with_differences
0,bulletin_file,258
1,cum_bulletin_file,258
2,season_year,0
3,state,0
4,date,0
5,season,0
6,fires_cumulative,0
7,fires_day,0
8,cum_period_start,0
9,day_source,0



MAHENDERGARH SUMMARY:


,records,available_daily_counts,available_cumulative_counts,available_day_sources,available_cum_sources,first_date,last_date
district,,,,,,,
Mahendergarh,258,0,258,0,258,2020-09-15,2025-10-27
Mahendragarh,258,0,258,0,258,2020-09-15,2025-10-27


### 🔎 Final Mahendragarh Source Verification

We have established that all 258 duplicate groups differ only
in their bulletin filenames.

Now we will verify that the numerical and validation fields
are identical and inspect the source filename differences.

No original records will be modified or deleted.

In [ ]:
# 1. Verify important numerical and validation columns

verification_columns = [
    "fires_day",
    "fires_cumulative",
    "cum_period_start",
    "day_source",
    "cum_source",
    "cum_check"
]

verification = (
    creams_duplicates
    .groupby(duplicate_keys, dropna=False)[verification_columns]
    .nunique(dropna=False)
)

print("ALL NUMERICAL AND VALIDATION VALUES IDENTICAL:",
      not verification.gt(1).any().any())

# 2. Inspect the different source filenames

source_comparison = (
    creams_duplicates.pivot(
        index=duplicate_keys,
        columns="district",
        values="bulletin_file"
    )
    .reset_index()
)

display(source_comparison.head(15))

# 3. Check cumulative bulletin filenames too

cum_source_comparison = (
    creams_duplicates.pivot(
        index=duplicate_keys,
        columns="district",
        values="cum_bulletin_file"
    )
    .reset_index()
)

display(cum_source_comparison.head(15))

ALL NUMERICAL AND VALIDATION VALUES IDENTICAL: True


district,state,district_standardized,date,season,season_year,Mahendergarh,Mahendragarh
0,Haryana,Mahendragarh,2020-09-15,rice,2020,01.RiceResidueFireBulletin_15Sep_2026_ICAR.pdf,01.RiceResidueFireBulletin_15Sep_2025_ICAR.pdf
1,Haryana,Mahendragarh,2020-09-16,rice,2020,02.RiceResidueFireBulletin_16Sep_2026_ICAR.pdf,02.RiceResidueFireBulletin_16Sep_2025_ICAR.pdf
2,Haryana,Mahendragarh,2020-09-17,rice,2020,03.RiceResidueFireBulletin_17Sep_2026_ICAR.pdf,03.RiceResidueFireBulletin_17Sep_2025_ICAR.pdf
3,Haryana,Mahendragarh,2020-09-18,rice,2020,04.RiceResidueFireBulletin_18Sep_2026_ICAR.pdf,04.RiceResidueFireBulletin_18Sep_2025_ICAR.pdf
4,Haryana,Mahendragarh,2020-09-19,rice,2020,05.RiceResidueFireBulletin_19Sep_2026_ICAR.pdf,05.RiceResidueFireBulletin_19Sep_2025_ICAR.pdf
5,Haryana,Mahendragarh,2020-09-20,rice,2020,06.RiceResidueFireBulletin_20Sep_2026_ICAR.pdf,06.RiceResidueFireBulletin_20Sep_2025_ICAR.pdf
6,Haryana,Mahendragarh,2020-09-21,rice,2020,07.RiceResidueFireBulletin_21Sep_2026_ICAR.pdf,07.RiceResidueFireBulletin_21Sep_2025_ICAR.pdf
7,Haryana,Mahendragarh,2020-09-22,rice,2020,08.RiceResidueFireBulletin_22Sep_2026_ICAR.pdf,08.RiceResidueFireBulletin_22Sep_2025_ICAR.pdf
8,Haryana,Mahendragarh,2020-09-23,rice,2020,09.RiceResidueFireBulletin_23Sep_2026_ICAR.pdf,09.RiceResidueFireBulletin_23Sep_2025_ICAR.pdf
9,Haryana,Mahendragarh,2020-09-24,rice,2020,10.RiceResidueFireBulletin_24Sep_2026_ICAR.pdf,10.RiceResidueFireBulletin_24Sep_2025_ICAR.pdf


district,state,district_standardized,date,season,season_year,Mahendergarh,Mahendragarh
0,Haryana,Mahendragarh,2020-09-15,rice,2020,01.RiceResidueFireBulletin_15Sep_2026_ICAR.pdf,01.RiceResidueFireBulletin_15Sep_2025_ICAR.pdf
1,Haryana,Mahendragarh,2020-09-16,rice,2020,02.RiceResidueFireBulletin_16Sep_2026_ICAR.pdf,02.RiceResidueFireBulletin_16Sep_2025_ICAR.pdf
2,Haryana,Mahendragarh,2020-09-17,rice,2020,03.RiceResidueFireBulletin_17Sep_2026_ICAR.pdf,03.RiceResidueFireBulletin_17Sep_2025_ICAR.pdf
3,Haryana,Mahendragarh,2020-09-18,rice,2020,04.RiceResidueFireBulletin_18Sep_2026_ICAR.pdf,04.RiceResidueFireBulletin_18Sep_2025_ICAR.pdf
4,Haryana,Mahendragarh,2020-09-19,rice,2020,05.RiceResidueFireBulletin_19Sep_2026_ICAR.pdf,05.RiceResidueFireBulletin_19Sep_2025_ICAR.pdf
5,Haryana,Mahendragarh,2020-09-20,rice,2020,06.RiceResidueFireBulletin_20Sep_2026_ICAR.pdf,06.RiceResidueFireBulletin_20Sep_2025_ICAR.pdf
6,Haryana,Mahendragarh,2020-09-21,rice,2020,07.RiceResidueFireBulletin_21Sep_2026_ICAR.pdf,07.RiceResidueFireBulletin_21Sep_2025_ICAR.pdf
7,Haryana,Mahendragarh,2020-09-22,rice,2020,08.RiceResidueFireBulletin_22Sep_2026_ICAR.pdf,08.RiceResidueFireBulletin_22Sep_2025_ICAR.pdf
8,Haryana,Mahendragarh,2020-09-23,rice,2020,09.RiceResidueFireBulletin_23Sep_2026_ICAR.pdf,09.RiceResidueFireBulletin_23Sep_2025_ICAR.pdf
9,Haryana,Mahendragarh,2020-09-24,rice,2020,10.RiceResidueFireBulletin_24Sep_2026_ICAR.pdf,10.RiceResidueFireBulletin_24Sep_2025_ICAR.pdf



### 🔍 Recreating Unpaired Mahendergarh Records

We identify records under the old district spelling that do not have a matching date under the standardized spelling.

These records will be preserved for investigation.

In [ ]:

# 1. Select records with the old spelling
mahendergarh_only = creams_ph[
    (creams_ph["state"] == "Haryana")
    & (creams_ph["district_standardized"] == "Mahendragarh")
    & (creams_ph["district"] == "Mahendergarh")
].copy()

# 2. Get dates available under the standardized spelling
mahendragarh_dates = set(
    creams_ph[
        (creams_ph["state"] == "Haryana")
        & (creams_ph["district"] == "Mahendragarh")
    ]["date"]
)

# 3. Keep old-spelling records with no matching date
unique_old_spelling = mahendergarh_only[
    ~mahendergarh_only["date"].isin(mahendragarh_dates)
].copy()

print("Old-spelling records:", len(mahendergarh_only))
print("Unpaired records:", len(unique_old_spelling))


Old-spelling records: 301
Unpaired records: 43


In [ ]:

print("SHAPE:")
print(unique_old_spelling.shape)

print("\nDATE RANGE:")
print("Start:", unique_old_spelling["date"].min())
print("End:", unique_old_spelling["date"].max())

print("\nSEASON DISTRIBUTION:")
print(unique_old_spelling["season"].value_counts(dropna=False))

print("\nAVAILABLE DAILY TARGETS:")
print(unique_old_spelling["fires_day"].notna().sum())

print("\nAVAILABLE CUMULATIVE TARGETS:")
print(unique_old_spelling["fires_cumulative"].notna().sum())

print("\nRECORDS BY YEAR:")
print(
    unique_old_spelling["date"]
    .dt.year.value_counts()
    .sort_index()
)

print("\nRECORD DETAILS:")
display(
    unique_old_spelling[
        [
            "state", "district", "date", "season", "season_year",
            "fires_day", "fires_cumulative",
            "bulletin_file", "cum_bulletin_file"
        ]
    ].sort_values("date")
)


SHAPE:
(43, 16)

DATE RANGE:
Start: 2026-09-15 00:00:00
End: 2026-10-27 00:00:00

SEASON DISTRIBUTION:
season
rice    43
Name: count, dtype: int64

AVAILABLE DAILY TARGETS:
0

AVAILABLE CUMULATIVE TARGETS:
43

RECORDS BY YEAR:
date
2026    43
Name: count, dtype: int64

RECORD DETAILS:


,state,district,date,season,season_year,fires_day,fires_cumulative,bulletin_file,cum_bulletin_file
133079,Haryana,Mahendergarh,2026-09-15,rice,2026,NaN,0.0,01.RiceResidueFireBulletin_15Sep_2026_ICAR.pdf,01.RiceResidueFireBulletin_15Sep_2026_ICAR.pdf
133311,Haryana,Mahendergarh,2026-09-16,rice,2026,NaN,0.0,02.RiceResidueFireBulletin_16Sep_2026_ICAR.pdf,02.RiceResidueFireBulletin_16Sep_2026_ICAR.pdf
133543,Haryana,Mahendergarh,2026-09-17,rice,2026,NaN,0.0,03.RiceResidueFireBulletin_17Sep_2026_ICAR.pdf,03.RiceResidueFireBulletin_17Sep_2026_ICAR.pdf
133775,Haryana,Mahendergarh,2026-09-18,rice,2026,NaN,0.0,04.RiceResidueFireBulletin_18Sep_2026_ICAR.pdf,04.RiceResidueFireBulletin_18Sep_2026_ICAR.pdf
134007,Haryana,Mahendergarh,2026-09-19,rice,2026,NaN,0.0,05.RiceResidueFireBulletin_19Sep_2026_ICAR.pdf,05.RiceResidueFireBulletin_19Sep_2026_ICAR.pdf
134239,Haryana,Mahendergarh,2026-09-20,rice,2026,NaN,0.0,06.RiceResidueFireBulletin_20Sep_2026_ICAR.pdf,06.RiceResidueFireBulletin_20Sep_2026_ICAR.pdf
134471,Haryana,Mahendergarh,2026-09-21,rice,2026,NaN,0.0,07.RiceResidueFireBulletin_21Sep_2026_ICAR.pdf,07.RiceResidueFireBulletin_21Sep_2026_ICAR.pdf
134703,Haryana,Mahendergarh,2026-09-22,rice,2026,NaN,0.0,08.RiceResidueFireBulletin_22Sep_2026_ICAR.pdf,08.RiceResidueFireBulletin_22Sep_2026_ICAR.pdf
134935,Haryana,Mahendergarh,2026-09-23,rice,2026,NaN,0.0,09.RiceResidueFireBulletin_23Sep_2026_ICAR.pdf,09.RiceResidueFireBulletin_23Sep_2026_ICAR.pdf
135167,Haryana,Mahendergarh,2026-09-24,rice,2026,NaN,0.0,10.RiceResidueFireBulletin_24Sep_2026_ICAR.pdf,10.RiceResidueFireBulletin_24Sep_2026_ICAR.pdf



### 🧹 Correcting Mahendragarh Duplicate Records

We will:

1. Identify old-spelling records (`Mahendergarh`).
2. Match them against canonical-spelling records (`Mahendragarh`) using state, standardized district, date, season and season year.
3. Remove only the 258 confirmed duplicate records.
4. Retain the 43 unmatched 2026 records.
5. Rebuild the provisional modelling base.

The original `creams_ph` DataFrame remains unchanged.


In [ ]:

# Define the duplicate matching keys
duplicate_keys = [
    "state",
    "district_standardized",
    "date",
    "season",
    "season_year"
]

# Select old-spelling records
old_records = creams_ph[
    (creams_ph["state"] == "Haryana")
    & (creams_ph["district"] == "Mahendergarh")
].copy()

# Preserve original row indexes
old_records["_original_index"] = old_records.index

# Select canonical-spelling records
canonical_records = creams_ph[
    (creams_ph["state"] == "Haryana")
    & (creams_ph["district"] == "Mahendragarh")
].copy()

# Get unique matching keys from canonical records
canonical_keys = canonical_records[duplicate_keys].drop_duplicates()

# Identify old-spelling records that have a confirmed matching key
matched_old = old_records.merge(
    canonical_keys,
    on=duplicate_keys,
    how="inner"
)

# Indexes of old-spelling records to remove
paired_old_indices = matched_old["_original_index"].tolist()

# Rebuild modelling base, removing only confirmed duplicate records
creams_model_base = creams_ph.drop(
    index=paired_old_indices
).copy()

# Verification
print("ORIGINAL CREAMS PH:", creams_ph.shape)
print("CONFIRMED DUPLICATE RECORDS REMOVED:", len(paired_old_indices))
print("MODELLING BASE:", creams_model_base.shape)

print("\nMAHENDRAGARH LABEL DISTRIBUTION:")
display(
    creams_model_base[
        (creams_model_base["state"] == "Haryana")
        & (creams_model_base["district_standardized"] == "Mahendragarh")
    ]["district"].value_counts()
)

print("\nREMAINING DUPLICATE RECORDS:")
print(
    creams_model_base.duplicated(
        subset=duplicate_keys,
        keep=False
    ).sum()
)


ORIGINAL CREAMS PH: (47655, 16)
CONFIRMED DUPLICATE RECORDS REMOVED: 258
MODELLING BASE: (47397, 16)

MAHENDRAGARH LABEL DISTRIBUTION:


district
Mahendragarh    1011
Mahendergarh      43
Name: count, dtype: int64


REMAINING DUPLICATE RECORDS:
0


In [ ]:

print("FIRMS COLUMNS:")
for col in fires_daily_integrated.columns:
    print("-", repr(col))

print("\nFIRST 5 RECORDS:")
display(fires_daily_integrated.head())

print("\nDATA TYPES:")
print(fires_daily_integrated.dtypes)


FIRMS COLUMNS:
- 'year'
- 'sensor_group'
- 'date'
- 'district_id'
- 'district'
- 'state'
- 'country'
- 'detections'
- 'boundary_id'

FIRST 5 RECORDS:


,year,sensor_group,date,district_id,district,state,country,detections,boundary_id
0,2003,modis,2003-11-19,7,Buxar,Bihar,India,1,7
1,2003,modis,2003-11-21,7,Buxar,Bihar,India,3,7
2,2003,modis,2003-11-26,7,Buxar,Bihar,India,2,7
3,2003,modis,2003-11-28,7,Buxar,Bihar,India,1,7
4,2003,modis,2003-11-30,7,Buxar,Bihar,India,2,7



DATA TYPES:
year                     int64
sensor_group            object
date            datetime64[ns]
district_id              int64
district                object
state                   object
country                 object
detections               int64
boundary_id              int64
dtype: object



### 🛰️ FIRMS Sensor Group Analysis

The FIRMS dataset uses `sensor_group` to identify satellite sensor categories.

We will inspect sensor distribution and check whether multiple sensor groups contribute detections for the same district on the same date.

This will help us decide the correct aggregation level before merging with CREAMS.


In [ ]:

print("FIRMS DAILY SHAPE:")
print(fires_daily_integrated.shape)

print("\nSENSOR GROUP DISTRIBUTION:")
print(fires_daily_integrated["sensor_group"].value_counts(dropna=False))

print("\nDATE RANGE:")
print("Start:", fires_daily_integrated["date"].min())
print("End:", fires_daily_integrated["date"].max())

print("\nUNIQUE DISTRICTS:")
print(fires_daily_integrated["boundary_id"].nunique())

# Check records per district and date
records_per_day = (
    fires_daily_integrated
    .groupby(["boundary_id", "date"])
    .size()
)

print("\nRECORDS PER DISTRICT-DATE:")
print(records_per_day.describe())

print("\nDISTRICT-DATE COMBINATIONS WITH MULTIPLE SENSOR GROUPS:")
print((records_per_day > 1).sum())

print("\nEXAMPLE MULTI-GROUP COMBINATIONS:")
display(
    records_per_day[records_per_day > 1]
    .head(10)
    .reset_index(name="sensor_group_records")
)


FIRMS DAILY SHAPE:
(427728, 9)

SENSOR GROUP DISTRIBUTION:
sensor_group
viirs    200690
snpp     153309
modis     73729
Name: count, dtype: int64

DATE RANGE:
Start: 2003-09-15 00:00:00
End: 2026-10-28 00:00:00

UNIQUE DISTRICTS:
852

RECORDS PER DISTRICT-DATE:
count    233411.000000
mean          1.832510
std           0.678762
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max           3.000000
dtype: float64

DISTRICT-DATE COMBINATIONS WITH MULTIPLE SENSOR GROUPS:
156822

EXAMPLE MULTI-GROUP COMBINATIONS:


,boundary_id,date,sensor_group_records
0,0,2012-11-07,2
1,0,2012-11-20,2
2,0,2013-11-10,2
3,0,2013-11-20,2
4,0,2013-11-22,2
5,0,2013-11-24,2
6,0,2015-11-13,2
7,0,2015-11-20,2
8,0,2016-10-16,2
9,0,2016-10-25,2



### 🌍 Filtering FIRMS to Punjab and Haryana

Our project covers only Punjab and Haryana, India.

Before aggregating sensor groups, we will inspect the country and state distribution and verify the district identifiers. This prevents records from other countries or states entering our modelling dataset.


In [ ]:

print("COUNTRY DISTRIBUTION:")
display(fires_daily_integrated["country"].value_counts(dropna=False).head(15))

print("\nSTATE DISTRIBUTION:")
display(fires_daily_integrated["state"].value_counts(dropna=False).head(20))

print("\nSENSOR GROUP DISTRIBUTION:")
display(fires_daily_integrated["sensor_group"].value_counts(dropna=False))

print("\nPUNJAB AND HARYANA RECORDS:")
fires_ph_check = fires_daily_integrated[
    (fires_daily_integrated["country"] == "India")
    & (fires_daily_integrated["state"].isin(["Punjab", "Haryana"]))
].copy()

print("Shape:", fires_ph_check.shape)

print("\nDISTRICT COUNTS BY STATE:")
display(
    fires_ph_check.groupby("state")["district"].nunique()
)

print("\nBOUNDARY ID CHECK:")
print("Missing IDs:", fires_ph_check["boundary_id"].isna().sum())
print("Unique IDs:", fires_ph_check["boundary_id"].nunique())
print("Zero IDs:", (fires_ph_check["boundary_id"] == 0).sum())

print("\nSAMPLE RECORDS:")
display(fires_ph_check.head())


COUNTRY DISTRIBUTION:


country
India       340855
Pakistan     86873
Name: count, dtype: int64


STATE DISTRIBUTION:


state
Punjab                98765
Uttar Pradesh         57335
Madhya Pradesh        33686
Haryana               25683
Maharashtra           20401
Sindh                 18952
Rajasthan             15971
Telangana             15939
Gujarat               14758
Karnataka             12457
Chhattisgarh          11520
Odisha                10056
Tamil Nadu             9974
Andhra Pradesh         9670
Khyber Pakhtunkhwa     9576
Jharkhand              8378
Assam                  7867
Bihar                  6441
West Bengal            6388
Uttarakhand            5807
Name: count, dtype: int64


SENSOR GROUP DISTRIBUTION:


sensor_group
viirs    200690
snpp     153309
modis     73729
Name: count, dtype: int64


PUNJAB AND HARYANA RECORDS:
Shape: (72410, 9)

DISTRICT COUNTS BY STATE:


state
Haryana    21
Punjab     22
Name: district, dtype: int64


BOUNDARY ID CHECK:
Missing IDs: 0
Unique IDs: 43
Zero IDs: 0

SAMPLE RECORDS:


,year,sensor_group,date,district_id,district,state,country,detections,boundary_id
109,2003,modis,2003-10-09,103,Ambala,Haryana,India,2,103
110,2003,modis,2003-10-12,103,Ambala,Haryana,India,1,103
111,2003,modis,2003-10-13,103,Ambala,Haryana,India,1,103
112,2003,modis,2003-10-14,103,Ambala,Haryana,India,2,103
113,2003,modis,2003-10-15,103,Ambala,Haryana,India,3,103



### 🛰️ Aggregating FIRMS Daily Satellite Detections

The FIRMS dataset can contain multiple sensor-group records for the same district and date.

We will transform it into one row per district-date.

Our output will contain:
- MODIS detections
- SNPP detections
- VIIRS detections
- Combined sensor detections
- Number of sensor groups reporting detections

We will preserve the original sensor-level dataset and avoid describing the combined total as unique fires.


In [ ]:

# Create a copy of the Punjab-Haryana FIRMS data
fires_ph = fires_ph_check.copy()

# Aggregate detections separately for each sensor group
fires_sensor_pivot = (
    fires_ph
    .pivot_table(
        index=["boundary_id", "district", "state", "date"],
        columns="sensor_group",
        values="detections",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Remove the columns index name
fires_sensor_pivot.columns.name = None

# Ensure all expected sensor columns exist
for sensor in ["modis", "snpp", "viirs"]:
    if sensor not in fires_sensor_pivot.columns:
        fires_sensor_pivot[sensor] = 0

# Rename sensor columns
fires_sensor_pivot = fires_sensor_pivot.rename(
    columns={
        "modis": "modis_detections",
        "snpp": "snpp_detections",
        "viirs": "viirs_detections"
    }
)

# Calculate combined detections
fires_sensor_pivot["total_sensor_detections"] = (
    fires_sensor_pivot["modis_detections"]
    + fires_sensor_pivot["snpp_detections"]
    + fires_sensor_pivot["viirs_detections"]
)

# Count sensor groups reporting detections
fires_sensor_pivot["reporting_sensor_groups"] = (
    fires_sensor_pivot[
        ["modis_detections", "snpp_detections", "viirs_detections"]
    ] > 0
).sum(axis=1)

# Verification
print("AGGREGATED FIRMS SHAPE:")
print(fires_sensor_pivot.shape)

print("\nCOLUMNS:")
print(fires_sensor_pivot.columns.tolist())

print("\nDUPLICATE DISTRICT-DATE KEYS:")
print(
    fires_sensor_pivot.duplicated(
        subset=["boundary_id", "date"]
    ).sum()
)

print("\nSENSOR DETECTION TOTALS:")
display(
    fires_sensor_pivot[
        [
            "modis_detections",
            "snpp_detections",
            "viirs_detections",
            "total_sensor_detections"
        ]
    ].sum()
)

print("\nREPORTING SENSOR GROUPS:")
display(
    fires_sensor_pivot["reporting_sensor_groups"].value_counts().sort_index()
)

print("\nSAMPLE:")
display(fires_sensor_pivot.head())


AGGREGATED FIRMS SHAPE:
(36581, 9)

COLUMNS:
['boundary_id', 'district', 'state', 'date', 'modis_detections', 'snpp_detections', 'viirs_detections', 'total_sensor_detections', 'reporting_sensor_groups']

DUPLICATE DISTRICT-DATE KEYS:
0

SENSOR DETECTION TOTALS:


modis_detections            324379
snpp_detections             866864
viirs_detections           1321096
total_sensor_detections    2512339
dtype: int64


REPORTING SENSOR GROUPS:


reporting_sensor_groups
1    12952
2    11429
3    12200
Name: count, dtype: int64


SAMPLE:


,boundary_id,district,state,date,modis_detections,snpp_detections,viirs_detections,total_sensor_detections,reporting_sensor_groups
0,103,Ambala,Haryana,2003-10-09,2,0,0,2,1
1,103,Ambala,Haryana,2003-10-12,1,0,0,1,1
2,103,Ambala,Haryana,2003-10-13,1,0,0,1,1
3,103,Ambala,Haryana,2003-10-14,2,0,0,2,1
4,103,Ambala,Haryana,2003-10-15,3,0,0,3,1



### 🔍 Validating Aggregated FIRMS Values

Before merging, we must verify that all sensor detection columns contain valid values.

We will check missing values in:
1. Original Punjab-Haryana FIRMS detections.
2. Aggregated sensor-specific columns.
3. Combined detection totals.

We will not replace missing values until we understand their source.


In [ ]:

print("MISSING VALUES IN ORIGINAL FIRMS DATA:")
print(fires_ph[["sensor_group", "detections"]].isna().sum())

print("\nMISSING VALUES AFTER AGGREGATION:")
print(
    fires_sensor_pivot[
        [
            "modis_detections",
            "snpp_detections",
            "viirs_detections",
            "total_sensor_detections",
            "reporting_sensor_groups"
        ]
    ].isna().sum()
)

print("\nROWS WITH ANY MISSING AGGREGATED VALUES:")
display(
    fires_sensor_pivot[
        fires_sensor_pivot[
            [
                "modis_detections",
                "snpp_detections",
                "viirs_detections",
                "total_sensor_detections",
                "reporting_sensor_groups"
            ]
        ].isna().any(axis=1)
    ].head(10)
)


MISSING VALUES IN ORIGINAL FIRMS DATA:
sensor_group    0
detections      0
dtype: int64

MISSING VALUES AFTER AGGREGATION:
modis_detections           0
snpp_detections            0
viirs_detections           0
total_sensor_detections    0
reporting_sensor_groups    0
dtype: int64

ROWS WITH ANY MISSING AGGREGATED VALUES:


,boundary_id,district,state,date,modis_detections,snpp_detections,viirs_detections,total_sensor_detections,reporting_sensor_groups



### 🔗 Preparing CREAMS District Identifiers

CREAMS identifies districts using state and standardized district names, whereas FIRMS uses numeric boundary IDs.

We will map CREAMS district names to the verified boundary IDs using our Punjab-Haryana boundary reference.

This will allow us to merge both datasets using a consistent district identifier and date.



### 🔍 Checking Boundary Dataset Column Names

Our boundary GeoDataFrame uses a different district-name column than expected.

We will inspect its actual column names and a sample of records before creating the district mapping.


In [ ]:

print("BOUNDARY COLUMNS:")
print(ph_boundaries.columns.tolist())

print("\nSAMPLE RECORDS:")
display(ph_boundaries.drop(columns="geometry").head())

print("\nCREAMS DISTRICT COLUMNS:")
print(creams_model_base.columns.tolist())


BOUNDARY COLUMNS:
['id', 'name', 'state', 'country', 'area_km2', 'geometry']

SAMPLE RECORDS:


,id,name,state,country,area_km2
103,103,Ambala,Haryana,India,1475.6
104,104,Bhiwani,Haryana,India,4613.7
105,105,Faridabad,Haryana,India,735.9
106,106,Fatehabad,Haryana,India,2427.6
107,107,Gurgaon,Haryana,India,1230.7



CREAMS DISTRICT COLUMNS:
['date', 'season', 'season_year', 'state', 'district', 'fires_day', 'fires_cumulative', 'cum_period_start', 'day_source', 'cum_source', 'cum_check', 'bulletin_file', 'cum_bulletin_file', 'district_standardized', 'geography_review_required', 'geography_status']



### 🗺️ Mapping CREAMS Districts to Boundary IDs

The boundary dataset stores district names in the `name` column.

We will map the 43 verified Punjab-Haryana districts to CREAMS using standardized district names and state names.

Newer districts without boundary polygons will remain unmatched and preserved.


In [ ]:

# 1. Create district mapping from verified boundaries
district_mapping = (
    ph_boundaries[
        ["state", "name", "id"]
    ]
    .copy()
    .rename(
        columns={
            "name": "district_standardized",
            "id": "boundary_id"
        }
    )
)

# 2. Normalize boundary district names
district_mapping["district_standardized"] = (
    district_mapping["district_standardized"].apply(normalize_name)
)

# 3. Prepare CREAMS copy
creams_merge_ready = creams_model_base.copy()

creams_merge_ready["district_standardized"] = (
    creams_merge_ready["district_standardized"].apply(normalize_name)
)

# 4. Map boundary IDs into CREAMS
creams_merge_ready = creams_merge_ready.merge(
    district_mapping,
    on=["state", "district_standardized"],
    how="left",
    validate="many_to_one"
)

# 5. Verify mapping
print("CREAMS MERGE-READY SHAPE:")
print(creams_merge_ready.shape)

print("\nMISSING BOUNDARY IDs:")
print(creams_merge_ready["boundary_id"].isna().sum())

print("\nUNMATCHED DISTRICTS:")
display(
    creams_merge_ready[
        creams_merge_ready["boundary_id"].isna()
    ][
        ["state", "district", "district_standardized", "geography_status"]
    ].drop_duplicates()
)

print("\nDISTRICT COVERAGE:")
display(
    creams_merge_ready.groupby("state")["boundary_id"].nunique()
)


CREAMS MERGE-READY SHAPE:
(47397, 17)

MISSING BOUNDARY IDs:
2153

UNMATCHED DISTRICTS:


,state,district,district_standardized,geography_status
24,Haryana,Charkhi Dadri,charkhi dadri,valid_district_no_boundary
10658,Punjab,Malerkotla,malerkotla,valid_district_no_boundary
10675,Haryana,Hansi,hansi,valid_district_no_boundary



DISTRICT COVERAGE:


state
Haryana    21
Punjab     22
Name: boundary_id, dtype: int64


### 🔗 Validating CREAMS and FIRMS Merge Keys

We will check whether CREAMS contains multiple records for the same district and date.

FIRMS has already been aggregated to one record per district-date.

Our objective is to ensure that joining the two datasets does not unexpectedly multiply rows or create duplicate observations.


In [ ]:

# Separate CREAMS records with valid boundary IDs
creams_with_boundary = creams_merge_ready[
    creams_merge_ready["boundary_id"].notna()
].copy()

print("CREAMS RECORDS WITH BOUNDARY IDs:")
print(creams_with_boundary.shape)

print("\nCREAMS DATE RANGE:")
print("Start:", creams_with_boundary["date"].min())
print("End:", creams_with_boundary["date"].max())

print("\nFIRMS DATE RANGE:")
print("Start:", fires_sensor_pivot["date"].min())
print("End:", fires_sensor_pivot["date"].max())

# Check duplicate district-date combinations in CREAMS
creams_key_counts = (
    creams_with_boundary
    .groupby(["boundary_id", "date"])
    .size()
)

print("\nCREAMS DISTRICT-DATE KEY CHECK:")
print("Total district-date combinations:", len(creams_key_counts))
print("Duplicate combinations:", (creams_key_counts > 1).sum())
print("Maximum records per district-date:", creams_key_counts.max())

# Check duplicate keys in FIRMS
print("\nFIRMS DISTRICT-DATE DUPLICATES:")
print(
    fires_sensor_pivot.duplicated(
        subset=["boundary_id", "date"]
    ).sum()
)


CREAMS RECORDS WITH BOUNDARY IDs:
(45244, 17)

CREAMS DATE RANGE:
Start: 2016-10-01 00:00:00
End: 2026-10-27 00:00:00

FIRMS DATE RANGE:
Start: 2003-09-19 00:00:00
End: 2026-10-28 00:00:00

CREAMS DISTRICT-DATE KEY CHECK:
Total district-date combinations: 45244
Duplicate combinations: 0
Maximum records per district-date: 1

FIRMS DISTRICT-DATE DUPLICATES:
0



### 🔗 Integrating CREAMS and FIRMS Daily Data

We will merge the CREAMS daily dataset with the aggregated FIRMS daily dataset using:

- `boundary_id`
- `date`

We will use a left join to preserve all 45,244 CREAMS records with valid boundary IDs.

We will also create a match indicator to distinguish matched FIRMS records from dates without a corresponding FIRMS record.

The original datasets will remain unchanged.


In [ ]:

# Perform the left merge
creams_fires_merged = creams_with_boundary.merge(
    fires_sensor_pivot,
    on=["boundary_id", "date"],
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_creams", "_firms")
)

# Create a match indicator
creams_fires_merged["firms_matched"] = (
    creams_fires_merged["_merge"] == "both"
)

# Verification
print("MERGED DATASET SHAPE:")
print(creams_fires_merged.shape)

print("\nMERGE STATUS:")
display(creams_fires_merged["_merge"].value_counts())

print("\nFIRMS MATCHED RECORDS:")
print(creams_fires_merged["firms_matched"].sum())

print("\nFIRMS UNMATCHED RECORDS:")
print((~creams_fires_merged["firms_matched"]).sum())

print("\nROW COUNT CHECK:")
print("Before merge:", len(creams_with_boundary))
print("After merge:", len(creams_fires_merged))

print("\nSAMPLE MERGED RECORDS:")
display(
    creams_fires_merged[
        [
            "date",
            "state_creams",
            "district_creams",
            "fires_day",
            "total_sensor_detections",
            "reporting_sensor_groups",
            "firms_matched"
        ]
    ].head(10)
)


MERGED DATASET SHAPE:
(45244, 26)

MERGE STATUS:


_merge
left_only     25793
both          19451
right_only        0
Name: count, dtype: int64


FIRMS MATCHED RECORDS:
19451

FIRMS UNMATCHED RECORDS:
25793

ROW COUNT CHECK:
Before merge: 45244
After merge: 45244

SAMPLE MERGED RECORDS:


,date,state_creams,district_creams,fires_day,total_sensor_detections,reporting_sensor_groups,firms_matched
0,2016-10-01,Punjab,Amritsar,NaN,87.0,3.0,True
1,2016-10-01,Punjab,Barnala,NaN,NaN,NaN,False
2,2016-10-01,Punjab,Bathinda,NaN,6.0,2.0,True
3,2016-10-01,Punjab,Faridkot,NaN,3.0,3.0,True
4,2016-10-01,Punjab,Fatehgarh Sahib,NaN,NaN,NaN,False
5,2016-10-01,Punjab,Fazilka,NaN,6.0,2.0,True
6,2016-10-01,Punjab,Firozpur,NaN,6.0,3.0,True
7,2016-10-01,Punjab,Gurdaspur,NaN,NaN,NaN,False
8,2016-10-01,Punjab,Hoshiarpur,NaN,NaN,NaN,False
9,2016-10-01,Punjab,Jalandhar,NaN,9.0,3.0,True



### 🔍 Post-Merge Data Quality Audit

We will audit the integrated CREAMS-FIRMS dataset.

We need to understand:

1. Daily target availability.
2. FIRMS feature availability.
3. Missing values in satellite detection columns.
4. Whether the merge introduced duplicate district-date records.
5. Distribution of FIRMS matches across Punjab and Haryana.

This audit will help us decide how to use satellite detections in our prediction models.


In [ ]:

print("DATASET SHAPE:")
print(creams_fires_merged.shape)

print("\nTARGET AVAILABILITY:")
print("Available fires_day:", creams_fires_merged["fires_day"].notna().sum())
print("Missing fires_day:", creams_fires_merged["fires_day"].isna().sum())

print("\nFIRMS FEATURE MISSINGNESS:")
firms_columns = [
    "modis_detections",
    "snpp_detections",
    "viirs_detections",
    "total_sensor_detections",
    "reporting_sensor_groups"
]

print(creams_fires_merged[firms_columns].isna().sum())

print("\nMATCH RATE BY STATE:")
display(
    creams_fires_merged.groupby("state_creams")["firms_matched"]
    .agg(["count", "sum", "mean"])
)

print("\nDUPLICATE DISTRICT-DATE KEYS:")
print(
    creams_fires_merged.duplicated(
        subset=["boundary_id", "date"]
    ).sum()
)

print("\nDAILY TARGET AND FIRMS MATCH CROSS-TAB:")
display(
    pd.crosstab(
        creams_fires_merged["firms_matched"],
        creams_fires_merged["fires_day"].notna(),
        margins=True
    )
)


DATASET SHAPE:
(45244, 26)

TARGET AVAILABILITY:
Available fires_day: 14235
Missing fires_day: 31009

FIRMS FEATURE MISSINGNESS:
modis_detections           25793
snpp_detections            25793
viirs_detections           25793
total_sensor_detections    25793
reporting_sensor_groups    25793
dtype: int64

MATCH RATE BY STATE:


,count,sum,mean
state_creams,,,
Haryana,22127,8176,0.369503
Punjab,23117,11275,0.487736



DUPLICATE DISTRICT-DATE KEYS:
0

DAILY TARGET AND FIRMS MATCH CROSS-TAB:


fires_day,False,True,All
firms_matched,,,
False,22344,3449,25793
True,8665,10786,19451
All,31009,14235,45244



### 🔍 Auditing Daily Target Count Consistency

We previously reported 14,392 available CREAMS daily targets.

The merged dataset currently contains 14,235.

We will compare target availability across each integration stage to identify where the difference occurred.

No records will be modified or deleted during this audit.


In [ ]:

datasets_to_check = {
    "Original CREAMS PH": creams_ph,
    "Corrected Modeling Base": creams_model_base,
    "CREAMS Merge Ready": creams_merge_ready,
    "CREAMS With Boundary": creams_with_boundary,
    "CREAMS + FIRMS Merged": creams_fires_merged
}

for name, data in datasets_to_check.items():
    print(
        f"{name}: "
        f"Rows = {len(data)}, "
        f"Available fires_day = {data['fires_day'].notna().sum()}, "
        f"Missing fires_day = {data['fires_day'].isna().sum()}"
    )

print("\nTARGETS IN OLD-SPELLING RECORDS:")
print(
    "Available daily targets:",
    old_records["fires_day"].notna().sum()
)

print("\nTARGETS IN CONFIRMED DUPLICATES REMOVED:")
print(
    "Available daily targets:",
    matched_old["fires_day"].notna().sum()
)

print("\nTARGETS IN 43 UNPAIRED RECORDS:")
print(
    "Available daily targets:",
    unique_old_spelling["fires_day"].notna().sum()
)


Original CREAMS PH: Rows = 47655, Available fires_day = 14392, Missing fires_day = 33263
Corrected Modeling Base: Rows = 47397, Available fires_day = 14392, Missing fires_day = 33005
CREAMS Merge Ready: Rows = 47397, Available fires_day = 14392, Missing fires_day = 33005
CREAMS With Boundary: Rows = 45244, Available fires_day = 14235, Missing fires_day = 31009
CREAMS + FIRMS Merged: Rows = 45244, Available fires_day = 14235, Missing fires_day = 31009

TARGETS IN OLD-SPELLING RECORDS:
Available daily targets: 0

TARGETS IN CONFIRMED DUPLICATES REMOVED:
Available daily targets: 0

TARGETS IN 43 UNPAIRED RECORDS:
Available daily targets: 0



### 🗺️ Audit of Districts Without Boundary Matches

Some valid Punjab and Haryana districts do not have matching polygons in our current boundary dataset.

We will:
1. Identify records without a boundary ID.
2. Count available daily targets in those records.
3. Break the results down by state and district.

We will preserve the original data and make no changes in this step.


In [ ]:

boundary_missing_records = creams_merge_ready[
    creams_merge_ready["boundary_id"].isna()
].copy()

print("RECORDS WITHOUT BOUNDARY MATCH:")
print("Total rows:", len(boundary_missing_records))

print(
    "Available fires_day:",
    boundary_missing_records["fires_day"].notna().sum()
)

print(
    "Missing fires_day:",
    boundary_missing_records["fires_day"].isna().sum()
)

print("\nDISTRICT-WISE BREAKDOWN:")

display(
    boundary_missing_records
    .groupby(["state", "district_standardized", "geography_status"])
    .agg(
        total_records=("date", "size"),
        available_targets=("fires_day", "count"),
        missing_targets=("fires_day", lambda x: x.isna().sum())
    )
    .reset_index()
    .sort_values(["state", "district_standardized"])
)


RECORDS WITHOUT BOUNDARY MATCH:
Total rows: 2153
Available fires_day: 157
Missing fires_day: 1996

DISTRICT-WISE BREAKDOWN:


,state,district_standardized,geography_status,total_records,available_targets,missing_targets
0,Haryana,charkhi dadri,valid_district_no_boundary,1042,37,1005
1,Haryana,hansi,valid_district_no_boundary,301,1,300
2,Punjab,malerkotla,valid_district_no_boundary,810,119,691



### 🔗 Complete CREAMS-FIRMS Integration

We will construct a complete integrated dataset for all valid Punjab and Haryana districts.

Districts without boundary matches will be retained, but their FIRMS satellite features will remain missing because we cannot reliably match them using the available district boundary identifiers.

This preserves the daily CREAMS target observations without inventing satellite detections.


In [ ]:

# 1. Separate boundary-matched and unmatched records

creams_boundary_part = creams_merge_ready[
    creams_merge_ready["boundary_id"].notna()
].copy()

creams_no_boundary_part = creams_merge_ready[
    creams_merge_ready["boundary_id"].isna()
].copy()


# 2. Merge boundary-matched records with FIRMS

merged_boundary_part = creams_boundary_part.merge(
    fires_sensor_pivot,
    on=["boundary_id", "date"],
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_creams", "_firms")
)


# 3. Keep districts without boundary matches

merged_no_boundary_part = creams_no_boundary_part.merge(
    fires_sensor_pivot.iloc[0:0],
    on=["boundary_id", "date"],
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("_creams", "_firms")
)


# 4. Combine both parts

creams_fires_merged_full = pd.concat(
    [merged_boundary_part, merged_no_boundary_part],
    ignore_index=True
)

creams_fires_merged_full["firms_matched"] = (
    creams_fires_merged_full["_merge"] == "both"
)


# 5. Verify the complete integration

print("COMPLETE MERGED DATASET:")
print("Shape:", creams_fires_merged_full.shape)

print("\nTARGET AVAILABILITY:")
print("Available:", creams_fires_merged_full["fires_day"].notna().sum())
print("Missing:", creams_fires_merged_full["fires_day"].isna().sum())

print("\nFIRMS MATCH STATUS:")
display(creams_fires_merged_full["_merge"].value_counts())

print("\nDISTRICT GEOGRAPHY STATUS:")
display(creams_fires_merged_full["geography_status"].value_counts())

print("\nROW COUNT CHECK:")
print("Expected rows:", len(creams_merge_ready))
print("Actual rows:", len(creams_fires_merged_full))

print("\nDUPLICATE VALIDATION:")
print(
    "Duplicate boundary-matched district-date keys:",
    creams_fires_merged_full.loc[
        creams_fires_merged_full["boundary_id"].notna()
    ].duplicated(subset=["boundary_id", "date"]).sum()
)


COMPLETE MERGED DATASET:
Shape: (47397, 26)

TARGET AVAILABILITY:
Available: 14392
Missing: 33005

FIRMS MATCH STATUS:


_merge
left_only     27946
both          19451
right_only        0
Name: count, dtype: int64


DISTRICT GEOGRAPHY STATUS:


geography_status
boundary_match                45244
valid_district_no_boundary     2153
Name: count, dtype: int64


ROW COUNT CHECK:
Expected rows: 47397
Actual rows: 47397

DUPLICATE VALIDATION:
Duplicate boundary-matched district-date keys: 0



### 💾 Saving the CREAMS-FIRMS Integrated Dataset

We have successfully integrated CREAMS daily burning observations with FIRMS satellite detections.

We will save this checkpoint in the processed data directory so it can be reused in subsequent integration and modeling notebooks.


In [ ]:

import os

PROCESSED_DIR = "../Data/processed_data"

INTEGRATED_DIR = os.path.join(
    PROCESSED_DIR,
    "integrated"
)

os.makedirs(INTEGRATED_DIR, exist_ok=True)

print("Integrated directory:", os.path.abspath(INTEGRATED_DIR))


Integrated directory: c:\Users\kenpa\OneDrive\Desktop\INT_324_Project\Data\processed_data\integrated


In [ ]:

INTEGRATED_DIR = os.path.join(PROCESSED_DIR, "integrated")

os.makedirs(INTEGRATED_DIR, exist_ok=True)

integration_path = os.path.join(
    INTEGRATED_DIR,
    "creams_fires_daily_integrated.csv"
)

creams_fires_merged_full.to_csv(
    integration_path,
    index=False
)

print("Saved successfully:")
print(integration_path)

print("\nSaved dataset shape:")
print(creams_fires_merged_full.shape)


Saved successfully:
../Data/processed_data\integrated\creams_fires_daily_integrated.csv

Saved dataset shape:
(47397, 26)


In [18]:
paddy_district = pd.read_csv(
    os.path.join(
        CLEANED_DIR,
        "paddy",
        "paddy_rice_area_district_clean.csv"
    )
)

In [19]:

paddy_integrated = paddy_district.copy()

paddy_integrated["lgd_id"] = pd.to_numeric(
    paddy_integrated["lgd_code"],
    errors="coerce"
)

paddy_integrated["district_standardized"] = (
    paddy_integrated.apply(
        lambda row: standardize_district(
            row["state"],
            row["district"]
        ),
        axis=1
    )
)

print("PADDY INTEGRATION DATASET CREATED")
print("Shape:", paddy_integrated.shape)

display(paddy_integrated.head())


PADDY INTEGRATION DATASET CREATED
Shape: (6674, 14)


,state,district,lgd_code,district_id,district_id_name,match_method,year,agri_year,kharif_area_ha,kharif_prod_t,total_area_ha,total_prod_t,lgd_id,district_standardized
0,Bihar,Araria,188,0,NaN,NaN,1997,NaN,111851.0,142460.0,120330,156453.0,188,Araria
1,Bihar,Araria,188,0,NaN,NaN,1998,NaN,93100.0,90299.0,103036,103912.0,188,Araria
2,Bihar,Araria,188,0,NaN,NaN,1999,NaN,24619.0,23238.0,38395,47965.0,188,Araria
3,Bihar,Araria,188,0,NaN,NaN,2000,NaN,112940.0,116192.0,127257,138801.0,188,Araria
4,Bihar,Araria,188,0,NaN,NaN,2001,NaN,96088.0,120503.0,125223,142114.0,188,Araria



### 🌾 Paddy Rice-Area Dataset Integration

Rice cultivation area is an important agricultural feature because crop residue burning is strongly associated with rice cultivation and harvesting.

Before merging this dataset, we will inspect its columns, geographic identifiers, and available agricultural years.

We will use LGD codes as the source district identifiers rather than assuming that legacy boundary IDs are equivalent.


In [20]:

print("PADDY DATASET SHAPE:")
print(paddy_integrated.shape)

print("\nCOLUMNS:")
print(paddy_integrated.columns.tolist())

print("\nDATA TYPES:")
print(paddy_integrated.dtypes)

print("\nSAMPLE RECORDS:")
display(paddy_integrated.head())

print("\nSTATE DISTRIBUTION:")
display(paddy_integrated["state"].value_counts(dropna=False))

print("\nDISTRICT COUNT:")
print(paddy_integrated["district"].nunique())

print("\nAGRICULTURAL YEAR AVAILABILITY:")
print(paddy_integrated["year"].value_counts(dropna=False).sort_index())


PADDY DATASET SHAPE:
(6674, 14)

COLUMNS:
['state', 'district', 'lgd_code', 'district_id', 'district_id_name', 'match_method', 'year', 'agri_year', 'kharif_area_ha', 'kharif_prod_t', 'total_area_ha', 'total_prod_t', 'lgd_id', 'district_standardized']

DATA TYPES:
state                     object
district                  object
lgd_code                   int64
district_id                int64
district_id_name         float64
match_method             float64
year                       int64
agri_year                float64
kharif_area_ha           float64
kharif_prod_t            float64
total_area_ha              int64
total_prod_t             float64
lgd_id                     int64
district_standardized     object
dtype: object

SAMPLE RECORDS:


,state,district,lgd_code,district_id,district_id_name,match_method,year,agri_year,kharif_area_ha,kharif_prod_t,total_area_ha,total_prod_t,lgd_id,district_standardized
0,Bihar,Araria,188,0,NaN,NaN,1997,NaN,111851.0,142460.0,120330,156453.0,188,Araria
1,Bihar,Araria,188,0,NaN,NaN,1998,NaN,93100.0,90299.0,103036,103912.0,188,Araria
2,Bihar,Araria,188,0,NaN,NaN,1999,NaN,24619.0,23238.0,38395,47965.0,188,Araria
3,Bihar,Araria,188,0,NaN,NaN,2000,NaN,112940.0,116192.0,127257,138801.0,188,Araria
4,Bihar,Araria,188,0,NaN,NaN,2001,NaN,96088.0,120503.0,125223,142114.0,188,Araria



STATE DISTRIBUTION:


state
Uttar Pradesh       2026
Madhya Pradesh      1232
Bihar               1054
Rajasthan            588
Punjab               562
Haryana              555
Uttarakhand          361
Himachal Pradesh     296
Name: count, dtype: int64


DISTRICT COUNT:
265

AGRICULTURAL YEAR AVAILABILITY:
year
1997    168
1998    231
1999    233
2000    235
2001    232
2002    231
2003    236
2004    237
2005    237
2006    238
2007    239
2008    242
2009    195
2010    244
2011    243
2012    248
2013    248
2014    243
2015    243
2016    235
2017    249
2018    251
2019    250
2020    252
2021    250
2022    252
2023    252
2024    260
Name: count, dtype: int64
